# レース自動精算 Colab

Google Drive上の未精算レースを公式結果ページと照合し、`results.csv` と `automation/` のログを更新します。


In [ ]:
import importlib
from pathlib import Path
import shutil
import subprocess
import sys

from google.colab import drive
from IPython.display import display

GITHUB_REPO = globals().get("GITHUB_REPO", "f94wzpjvr2-stack/keiba-analysis")
GITHUB_BRANCH = globals().get("GITHUB_BRANCH", "main")
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/keiba-analysis")
DRIVE_DATA_DIR = globals().get("DRIVE_DATA_DIR", "/content/drive/MyDrive/keiba-ev-data")
REQUEST_INTERVAL = 2.0

drive.mount("/content/drive")
project = Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
    f"https://github.com/{GITHUB_REPO}.git", PROJECT_DIR,
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_DIR}[dev]"
], check=True)
src_dir = f"{PROJECT_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

settlement = importlib.import_module("keiba_ev.settlement")
reporting = importlib.import_module("keiba_ev.performance_report")

result = settlement.settle_unsettled_races(
    DRIVE_DATA_DIR,
    request_interval=REQUEST_INTERVAL,
)
summary = result["summary"]
results = result["results"]
settled = result["settled"]
report = reporting.build_performance_report(results, settled)

print("今回精算したレース数:", summary["settled_races"])
print("今回の投資額:", report["current"]["stake"])
print("今回の払戻額:", report["current"]["payout"])
print("今回の利益:", report["current"]["profit"])
print("今回の回収率:", report["current"]["return_rate"])
print("累積投資額:", report["cumulative"]["stake"])
print("累積払戻額:", report["cumulative"]["payout"])
print("累積利益:", report["cumulative"]["profit"])
print("累積回収率:", report["cumulative"]["return_rate"])
print("スキップ:", summary["skipped_races"], "エラー:", summary["errors"])

print("券種別回収率")
display(report["by_bet_type"])
print("algorithm_version別回収率")
display(report["by_algorithm_version"])

print("保存ファイル")
for path in sorted(Path(DRIVE_DATA_DIR).glob("*.csv")):
    print(" -", path.name)
for path in sorted((Path(DRIVE_DATA_DIR) / "automation").glob("*.csv")):
    print(" - automation/" + path.name)
